In [ ]:
# Databricks notebook source
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

import os

# ---------------------------------------------------------------------------
# Externalized configuration (Databricks widgets / environment variables).
# ---------------------------------------------------------------------------
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "landing")
dbutils.widgets.text("catalog", "job_offers")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("export_prefix", "gold/web_export")


def _cfg(name: str, default: str = "") -> str:
    """Resolve a parameter: widget -> environment variable -> default."""
    value = dbutils.widgets.get(name)
    return value if value else os.environ.get(name.upper(), default)


STORAGE_ACCOUNT = _cfg("storage_account")
CONTAINER = _cfg("container", "landing")
CATALOG = _cfg("catalog", "job_offers")
GOLD_SCHEMA = _cfg("gold_schema", "gold")
EXPORT_PREFIX = _cfg("export_prefix", "gold/web_export")

if not STORAGE_ACCOUNT:
    raise ValueError(
        "Missing required parameter 'storage_account': set the storage account "
        "that holds the export container.")


# Web export (Gold -> Parquet + meta.json)

Builds the export consumed by the static web dashboard (`docs/dashboard`):
reads the Gold tables, projects them to the public contract (exact columns
and geography ported from Power BI) and writes one Parquet per table plus
`meta.json` (freshness, counts, size and mode) to ADLS (`abfss://`).

Runs at the end of the daily job, after `gold_build`. It does not write raw
data, descriptions or internal columns: public fields only. The cluster
provides the credential (no SAS or tokens in the notebook).


In [ ]:
# Gold tables already built: the notebook only projects (no dedup).
fact_offers = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_offers")
fact_offer_skills = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_offer_skills")
dim_skill_list = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_skill_list")
dim_calendar = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_calendar")


In [ ]:
# Sibling module in the repo (same pattern as Prepare_Gold in gold_build.ipynb).
import Prepare_Web_Export as web_export

dest = (f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
        f"{EXPORT_PREFIX.strip('/')}")

tables = web_export.build_web_export(
    spark, fact_offers, fact_offer_skills, dim_skill_list, dim_calendar)

table_counts, sources, data_date = web_export.collect_export_stats(tables)
meta = web_export.build_meta(table_counts, sources, data_date)

# size_provider is called inside write_web_export after writing the four
# Parquet files and before meta.json. Only .parquet files are counted: on
# re-runs the previous meta.json stays in dest and must not count.
size_provider = lambda: sum(
    f.size for f in dbutils.fs.ls(dest) if f.name.endswith(".parquet"))

updated_meta = web_export.write_web_export(
    spark, tables, dest, meta=meta, size_provider=size_provider)


In [ ]:
size_mb = round(updated_meta["size_bytes"] / 1024 / 1024, 2)

print("Web export: summary")
print(f"  destination: {dest}")
print(f"  mode:        {updated_meta['mode']}")
print(f"  size:        {updated_meta['size_bytes']} bytes ({size_mb} MB)")
print(f"  data date:   {updated_meta['data_date']}")
print(f"  generated:   {updated_meta['generated_at']}")
print(f"  tables:      {updated_meta['tables']}")
print(f"  sources:     {updated_meta['sources']}")

display(spark.createDataFrame(
    [("dest", dest),
     ("mode", updated_meta["mode"]),
     ("size_bytes", str(updated_meta["size_bytes"])),
     ("size_mb", str(size_mb)),
     ("data_date", str(updated_meta["data_date"])),
     ("generated_at", updated_meta["generated_at"])],
    ["field", "value"]))
